# BERT Sentiment Classification — Baseline vs. Class-Weighted (RQ2)

Runs `src/train.py` + `src/evaluate.py` twice — once with `use_class_weights: false` (baseline) and once with `use_class_weights: true` (RQ2 class-balancing experiment) — then compares the neutral-class F1/recall between the two.

Works on any CUDA GPU (Colab, Kaggle, a lab machine, your own workstation, a rented cloud instance, etc.) — nothing here is Colab-specific. Just make sure a GPU is attached/selected before running, then run all cells top to bottom.

In [ ]:
# This notebook lives at experiments/bert_class_balancing.ipynb, one
# level below the repo root, and Jupyter starts a notebook's cwd at
# its own directory -- so cwd here is .../experiments, not the repo
# root. Every cell after this one (pip install, python src/train.py,
# etc.) uses paths relative to the repo root, so cd up first.
import os

if not os.path.isfile("configs/bert_config.yaml") and os.path.isfile("../configs/bert_config.yaml"):
    %cd ..

# Assumes the repo is already cloned and this notebook is being run
# from inside it (e.g. a persistent GPU machine/cluster you manage
# yourself). Just makes sure you're on the right branch and up to date.
# Once PR #8 (https://github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis/pull/8)
# is merged, change BRANCH to "main".
BRANCH = "feat/bert-class-weighting"

!git checkout {BRANCH}

# A previous run of this notebook may have left configs/bert_config.yaml
# locally modified (see set_class_weights below) -- stash it first so
# `git pull` can't silently fail/conflict against an uncommitted change.
status = !git status --porcelain configs/bert_config.yaml
if status:
    print("Local changes to configs/bert_config.yaml detected -- stashing before pull.")
    !git stash push -- configs/bert_config.yaml

!git pull
!pwd && ls

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- attach/select one before continuing.")

## 1. Baseline run (`use_class_weights: false`)

This is already the default in `configs/bert_config.yaml`, but the next cell sets it explicitly so this notebook is safe to re-run from the top.

In [ ]:
import re

CONFIG_PATH = "configs/bert_config.yaml"


def set_class_weights(enabled: bool) -> None:
    """Flip use_class_weights via a targeted line replacement rather than
    a full yaml.safe_load/safe_dump round-trip, which would silently
    strip every comment in the file (including the RQ2 explanation right
    above this same setting)."""
    text = open(CONFIG_PATH).read()
    new_text, count = re.subn(
        r"^(\s*use_class_weights:[ \t]*)(true|false)[ \t]*$",
        lambda m: f"{m.group(1)}{'true' if enabled else 'false'}",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    if count == 0:
        raise RuntimeError(
            f"Could not find a 'use_class_weights: true|false' line in {CONFIG_PATH}"
        )
    open(CONFIG_PATH, "w").write(new_text)
    print(f"use_class_weights set to {enabled}")


set_class_weights(False)

In [ ]:
!python src/train.py

In [ ]:
!python src/evaluate.py

## 2. Weighted run (`use_class_weights: true`)

In [ ]:
set_class_weights(True)

In [ ]:
!python src/train.py

In [ ]:
!python src/evaluate.py

## 3. Compare baseline vs. weighted (RQ2 answer)

Reads the structured JSON results both runs saved to `outputs/` and compares them directly — the neutral-class row is the one RQ2 cares about. A `NaN` for a metric means that class was never predicted at all (undefined), which is distinct from a real `0.0` (predicted, but always wrong).

In [ ]:
import json

with open("outputs/evaluation_results_baseline.json") as f:
    baseline = json.load(f)

with open("outputs/evaluation_results_weighted.json") as f:
    weighted = json.load(f)

print(f"{'Metric':<24}{'Baseline':>12}{'Weighted':>12}{'Delta':>12}")
print("-" * 60)


def row(label, b, w):
    print(f"{label:<24}{b:>12.4f}{w:>12.4f}{w - b:>+12.4f}")


row("Accuracy", baseline["accuracy"], weighted["accuracy"])
row("Precision Macro", baseline["precision_macro"], weighted["precision_macro"])
row("Recall Macro", baseline["recall_macro"], weighted["recall_macro"])
row("F1 Macro", baseline["f1_macro"], weighted["f1_macro"])
row("F1 Weighted", baseline["f1_weighted"], weighted["f1_weighted"])
print()
for sentiment in ["negative", "neutral", "positive"]:
    row(
        f"Precision ({sentiment})",
        baseline["precision_per_class"][sentiment],
        weighted["precision_per_class"][sentiment],
    )
    row(
        f"Recall ({sentiment})",
        baseline["recall_per_class"][sentiment],
        weighted["recall_per_class"][sentiment],
    )
    row(
        f"F1 ({sentiment})",
        baseline["f1_per_class"][sentiment],
        weighted["f1_per_class"][sentiment],
    )

print()
neutral_f1_delta = weighted["f1_per_class"]["neutral"] - baseline["f1_per_class"]["neutral"]
neutral_recall_delta = weighted["recall_per_class"]["neutral"] - baseline["recall_per_class"]["neutral"]
print(f"Neutral-class F1 delta:     {neutral_f1_delta:+.4f}")
print(f"Neutral-class recall delta: {neutral_recall_delta:+.4f}")
if neutral_f1_delta > 0:
    print(f"RQ2: class weighting IMPROVED neutral-class F1 by {neutral_f1_delta:+.4f}")
elif neutral_f1_delta < 0:
    print(f"RQ2: class weighting HURT neutral-class F1 by {neutral_f1_delta:+.4f}")
else:
    print("RQ2: class weighting made no difference to neutral-class F1")

In [ ]:
import pandas as pd

labels = ["negative", "neutral", "positive"]

print("Baseline confusion matrix (rows=true, cols=predicted):")
display(pd.DataFrame(baseline["confusion_matrix"], index=labels, columns=labels))

print("\nWeighted confusion matrix (rows=true, cols=predicted):")
display(pd.DataFrame(weighted["confusion_matrix"], index=labels, columns=labels))

## 4. Commit the results

If you're running this on a persistent machine (your own GPU box, a server you SSH into, etc.), the repo and its `.git` history are already right here — just commit normally:
```bash
git add outputs/
git commit -m "Add baseline/weighted BERT evaluation results"
git push
```

If instead you're on an ephemeral cloud notebook (Colab/Kaggle) where the filesystem disappears when the session ends, either download `outputs/` via the file browser before disconnecting, or push directly from here:

In [ ]:
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add outputs/
# !git commit -m "Add baseline/weighted BERT evaluation results"
# !git push https://<github-token>@github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis.git {BRANCH}